# Lotte Reiniger Style — SDXL LoRA in Colab

Generates images in the silhouette-animation style of Lotte Reiniger using the [`KappaNeuro/lotte-reiniger-style`](https://huggingface.co/KappaNeuro/lotte-reiniger-style) LoRA on top of Stable Diffusion XL 1.0.

**Before you run:** set the runtime to GPU (Runtime → Change runtime type → T4 / L4 / A100).

The trigger phrase is **`Lotte Reiniger Style`** — put it at the start of every prompt.

## 1. Install dependencies

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate safetensors peft huggingface_hub "torchao>=0.16"

## 2. Load SDXL base + the Lotte Reiniger LoRA

In [ ]:
import torch
from diffusers import DiffusionPipeline
from huggingface_hub import hf_hub_download, list_repo_files
from safetensors.torch import load_file

pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
).to("cuda")

# KappaNeuro stores the LoRA under an arbitrary filename, so auto-discover it.
# We also drop text-encoder keys to avoid an IndexError in current diffusers/PEFT.
repo = "KappaNeuro/lotte-reiniger-style"
files = list_repo_files(repo)
lora_file = next(f for f in files if f.endswith(".safetensors"))
print("Loading LoRA:", lora_file)

lora_path = hf_hub_download(repo, lora_file)
state_dict = load_file(lora_path)
unet_sd = {k: v for k, v in state_dict.items()
           if "text_encoder" not in k and "lora_te" not in k}
pipe.load_lora_weights(unet_sd)

# Uncomment if you hit OOM on a free T4:
# pipe.enable_model_cpu_offload()

## 3. Generate an image

In [ ]:
prompt = (
    "Lotte Reiniger Style - a princess and a stag in a moonlit forest, "
    "intricate paper-cutout silhouette, ornate filigree, full-moon backlight"
)
negative_prompt = "photorealistic, color photo, low contrast, blurry"

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=30,
    guidance_scale=7.0,
    height=1024,
    width=1024,
).images[0]

image.save("lotte_reiniger_output.png")
image

## 4. Tweak the LoRA strength (optional)

If the style is too overpowering or too faint, scale the adapter:

In [ ]:
active = pipe.get_active_adapters()
adapter_name = list(active)[0] if active else "default_0"
pipe.set_adapters([adapter_name], adapter_weights=[0.8])

image = pipe(
    prompt="Lotte Reiniger Style - sleeping beauty in a thorned castle, silhouette, candlelight",
    num_inference_steps=30,
    guidance_scale=7.0,
).images[0]
image

## 5. Batch a few prompts

In [ ]:
prompts = [
    "Lotte Reiniger Style - Prince Achmed flying on a magical horse over Arabian palaces",
    "Lotte Reiniger Style - a witch stirring a cauldron in a gnarled forest at night",
    "Lotte Reiniger Style - a swan princess by a moonlit lake, art nouveau filigree",
]

for i, p in enumerate(prompts):
    img = pipe(p, num_inference_steps=30, guidance_scale=7.0).images[0]
    img.save(f"out_{i:02d}.png")
    display(img)